# 多项式表示与拟合

学习目标：用 Polynomial 表示、计算和拟合多项式，解释系数与区间映射，检查拟合误差及高次拟合的限制。

前置知识：数组、函数、最小二乘、导数与定积分的基本含义。

运行环境：Python 3.12、NumPy 2.5。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

本章输入均为自制数据，后续单元沿用导入的 np 和 Polynomial。

## 1 多项式与求值

Polynomial 用从低次到高次排列的系数表示多项式。Polynomial([1, 2, 1]) 表示 1 + 2x + x²，其中 x 是代入的数值。把数组传给这个对象，就能一次计算多个位置的函数值。

先计算三个位置的结果，并与熟悉的 (x + 1)² 对照。

In [1]:
import numpy as np
from numpy.polynomial import Polynomial

p = Polynomial([1, 2, 1])
x = np.array([0.0, 1.0, 2.0])
print(p(x))  # [1. 4. 9.]
print((x + 1) ** 2)  # [1. 4. 9.]，与 p(x) 一致。
print(p.coef, p.degree())  # 系数由常数项开始，次数为 2。

[1. 4. 9.]
[1. 4. 9.]
[1. 2. 1.] 2


## 2 系数与多项式运算

没有出现的中间次数仍要保留系数位置。例如 1 + 3x² 写成 [1, 0, 3]。默认 domain 和 window 都是 [-1, 1]，此时系数对应原变量 x 的幂；自定义区间或拟合结果的系数需要结合映射解释。

多项式对象可以相加、相减、相乘；这些操作得到新的多项式，不是只对系数逐元素计算。

In [2]:
quadratic = Polynomial([1, 0, 3])
linear = Polynomial([1, 2])
product = quadratic * linear
print(quadratic(np.array([0, 1, 2])))  # [1. 4. 13.]
print(product.coef)  # (1 + 3x²)(1 + 2x) = 1 + 2x + 3x² + 6x³。
print(product(2), quadratic(2) * linear(2))  # 65.0 65.0

[ 1.  4. 13.]
[1. 2. 3. 6.]
65.0 65.0


## 3 导数与积分

deriv(m) 返回 m 阶导数，默认 m=1；integ() 返回一次积分得到的多项式，默认积分常数为 0。两者仍然可以像原多项式一样求值。

这里对 1 + 2x + x² 求导和积分，用积分多项式在上下限的差计算从 0 到 1 的定积分。

In [3]:
p = Polynomial([1, 2, 1])
derivative = p.deriv()
antiderivative = p.integ()
print(derivative.coef, p.deriv(2).coef)  # 一阶导数 2 + 2x；二阶导数 2。
print(antiderivative.coef)  # x + x² + x³ / 3。
integral = antiderivative(1) - antiderivative(0)
print(integral, np.isclose(integral, 7 / 3))  # 约 2.333333333333333，True。
print(derivative(np.array([0, 1, 2])))  # [2. 4. 6.]

[2. 2.] [2.]
[0.         1.         1.         0.33333333]
2.333333333333333 True
[2. 4. 6.]


integ(k=..., lbnd=...) 可以指定积分多项式在下限 lbnd 处的值。k 是积分常数参数；一次积分时，下面的结果在 x=2 处等于 5。对于带区间映射的对象，deriv() 和 integ() 会考虑映射尺度，保留其 domain 和 window。

In [4]:
anchored = p.integ(k=5, lbnd=2)
print(anchored(2))  # 5.0。
print(np.allclose(anchored.deriv().coef, p.coef))  # 积分后求导恢复原系数。

5.0
True


## 4 最小二乘拟合

Polynomial.fit(x, y, deg) 根据观测位置 x 和观测值 y，返回指定次数 deg 的最小二乘拟合多项式。次数为 1 表示拟合直线。

先用已知关系 y = 1 + 2x 生成四个点。浮点计算可能留下微小误差，因此用容差检查，而不是要求系数和结果逐位相同。本章小规模 float64 示例采用绝对容差 1e-12、相对容差 0；它不是任意拟合任务的通用阈值。

In [5]:
x = np.array([10.0, 11.0, 12.0, 13.0])
y = 1 + 2 * x
fitted = Polynomial.fit(x, y, deg=1)
prediction = fitted(x)
print(prediction)  # 对应 [21. 23. 25. 27.]。
print(np.allclose(prediction, y, rtol=0, atol=1e-12))  # True
print(np.max(np.abs(prediction - y)))  # 观察本次计算的最大绝对误差。

[21. 23. 25. 27.]
True
3.552713678800501e-15


## 5 domain 与 window

domain 是输入变量的映射区间，window 是映射后的区间。fit() 默认用覆盖输入位置的最小区间作为 domain，Polynomial 的默认 window 是 [-1, 1]。

前面 x 在 [10, 13] 中，内部变量为 t = (x − 11.5) / 1.5。把三个代表位置放到两条数轴上，能看出为什么 fitted.coef 不能直接解释为 x 的幂系数。

![输入坐标 10、11.5、13 分别映射到内部坐标 -1、0、1，24+3t 与 1+2x 描述相同的求值。](image/illustration/24-01-domain-window.svg)

拟合对象的系数约 [24, 3] 表示 24 + 3t，但 fitted(x) 仍接收原坐标 x，并自动完成映射。图中只展示本例线性映射，domain 不表示对象会禁止区间外求值。

convert() 不指定参数时转换到 Polynomial 的默认区间，此时系数约 [1, 2]，可解释为 1 + 2x。下面先检查两套系数及求值一致性，再观察忽略映射、直接重建 Polynomial(fitted.coef) 会怎样改变结果。

In [6]:
print(fitted.domain, fitted.window)  # [10. 13.] [-1. 1.]
print(fitted.coef)  # 约 [24. 3.]，对应映射后的变量。
ordinary = fitted.convert()
print(ordinary.coef)  # 约 [1. 2.]，对应原变量。
print(np.allclose(ordinary(x), fitted(x), rtol=0, atol=1e-12))  # True
wrong = Polynomial(fitted.coef)
print(wrong(x))  # [54. 57. 60. 63.]；误把 24 + 3t 当成 24 + 3x。

[10. 13.] [-1.  1.]
[24.  3.]
[1. 2.]
True
[54. 57. 60. 63.]


domain 不会把超出区间的输入自动截断。下面 [10, 20] 映射到 [-1, 1]，输入 25 会映射为 2。多项式能计算区间外的值，并不代表对观测数据的外推可信。

In [7]:
mapped = Polynomial([0, 1], domain=[10, 20])
positions = np.array([10.0, 15.0, 20.0, 25.0])
print(mapped(positions))  # [-1. 0. 1. 2.]。
print(mapped.convert().coef)  # [-3. 0.2]，即 -3 + 0.2x。
print(mapped.deriv()(positions))  # 对原变量求导，均为 0.2。

[-1.  0.  1.  2.]
[-3.   0.2]
[0.2 0.2 0.2 0.2]


## 6 拟合误差与诊断

带噪声的观测通常不能被低次多项式完全通过。残差是观测值减去拟合值；残差平方和衡量这些位置上的拟合误差，不能单独说明模型在新位置的表现。

full=True 另外返回诊断列表，依次是残差平方和、拟合矩阵的数值秩、缩放后拟合矩阵的奇异值以及用于判断数值秩的 rcond。这里次数为 1，待估计系数共有 2 个。

In [8]:
x_observed = np.array([0.0, 1.0, 2.0, 3.0])
y_observed = np.array([1.0, 2.1, 2.9, 4.2])
line, diagnostics = Polynomial.fit(x_observed, y_observed, deg=1, full=True)
residuals, rank, singular_values, rcond = diagnostics
errors = y_observed - line(x_observed)
print(errors)  # 约 [0.01, 0.07, -0.17, 0.09]。
print(np.sum(errors ** 2), residuals)  # 残差平方和约 0.042。
print(rank, singular_values, rcond)  # 秩为 2，可估计两个线性系数。
print(np.isclose(np.sum(errors ** 2), residuals[0]))  # True

[ 0.01  0.07 -0.17  0.09]
0.041999999999999996 [0.042]
2 [1. 1.] 8.881784197001252e-16
True


## 7 次数选择与拟合风险

增加次数可能减小观测位置的误差，却不能因此确定真实关系。高次幂拟合还可能受到病态问题和舍入误差影响；合适的区间映射能改善部分数值条件，但不能保证拟合可靠。

仍用四个观测点，比较一次和三次拟合。三次多项式可以几乎经过这四个点，但两者在 x=4 的预测明显不同。这里没有 x=4 的真实观测，不能凭训练误差决定哪个预测正确。

In [9]:
cubic = Polynomial.fit(x_observed, y_observed, deg=3)
print(np.sum((y_observed - line(x_observed)) ** 2))  # 约 0.042。
print(np.sum((y_observed - cubic(x_observed)) ** 2))  # 接近 0，不代表发现真实规律。
print(line(4), cubic(4))  # 约 5.15 与 6.8，仅为这组数据的两种外推结果。

0.041999999999999996
5.5220263365470826e-30
5.15 6.799999999999999


当数据不足以确定全部系数，或者数值秩不足时，应检查诊断，而不是把“返回了对象”视作拟合可靠。下面用四个观测点拟合五次多项式，共有六个系数，数值秩最多为 4。

full=True 时通过诊断读取问题；full=False 时，秩不足可能触发 RankWarning。欠定或秩不足情况下，诊断中的残差数组可能为空；空数组不等于残差平方和为 0，必要时直接计算观测残差。

In [10]:
high_degree, diagnostics = Polynomial.fit(x_observed, y_observed, deg=5, full=True)
residuals, rank, singular_values, rcond = diagnostics
print(rank, len(high_degree.coef))  # 4 与 6，不能唯一确定全部系数。
print(residuals, residuals.size)  # 诊断残差数组为空。
print(np.sum((y_observed - high_degree(x_observed)) ** 2))  # 接近 0，不代表系数唯一。
print(high_degree(4))  # 仍能返回外推值，不能据此认为解可靠。

4 6
[] 0
1.1832913578315177e-30
7.171330907648972


## 8 选学：根与其他多项式族
roots() 返回多项式的根。根可能是复数；数值求根也会产生误差，应把得到的根代回多项式检查。重复根等情况可能更敏感，不能仅根据打印出的有限小数位判断精度。

In [11]:
root_example = Polynomial([-1, 0, 1])
roots = root_example.roots()
print(roots)  # x² - 1 的根为 -1 和 1。
print(root_example(roots))  # [0. 0.]，两个根代回为零。
print(np.allclose(root_example(roots), 0, rtol=0, atol=1e-12))  # 预期：True，求得的根代回后在给定容差内为零。

[-1.  1.]
[0. 0.]
True


numpy.polynomial 还提供不同的多项式基。同一串系数在不同基下含义不同；下面只认识名称和转换入口，不展开正交性的证明。

| 名称 | 中文名称／含义 |
| --- | --- |
| Chebyshev | 切比雪夫多项式 |
| Legendre | 勒让德多项式 |
| Hermite | 物理学家形式的埃尔米特多项式 |
| HermiteE | 概率学家形式的埃尔米特多项式 |
| Laguerre | 拉盖尔多项式 |

Chebyshev([0, 0, 1]) 表示二次切比雪夫基函数，即 2x² − 1，而不是 x²。用 convert(kind=Polynomial) 转成幂基表示后，才比较幂的系数。

In [12]:
from numpy.polynomial import Chebyshev

chebyshev = Chebyshev([0, 0, 1])
power_basis = chebyshev.convert(kind=Polynomial)
points = np.array([-1.0, 0.0, 1.0])
print(power_basis.coef)  # [-1. 0. 2.]。
print(chebyshev(points), power_basis(points))  # 均为 [1. -1. 1.]。
print(Polynomial([0, 0, 1])(points))  # 幂基的相同系数表示 x²。

[-1.  0.  2.]
[ 1. -1.  1.] [ 1. -1.  1.]
[1. 0. 1.]


## 9 选学：识别旧接口
旧接口 poly1d 把系数从高次到低次排列，恰好与 Polynomial 相反。阅读已有代码时先确认使用的是哪个接口；新代码优先使用 numpy.polynomial 提供的类。

In [13]:
legacy = np.poly1d([1, 2, 3])
modern = Polynomial([1, 2, 3])
print(legacy(2))  # x² + 2x + 3 在 2 处为 11。
print(modern(2))  # 1 + 2x + 3x² 在 2 处为 17。
print(Polynomial([3, 2, 1])(2))  # 反转旧系数后才表示相同的幂基多项式。

11
17.0
11.0


## 本章小结

（1）Polynomial 的幂基系数从常数项开始；求值、求导和积分返回的结果都要结合变量含义理解。

（2）fit() 返回的系数对应 domain 到 window 的映射变量。需要原变量的幂基系数时使用 convert().coef。

（3）拟合应检查观测误差和诊断信息。高次拟合、秩不足和外推需要额外判断，不能把通过样本当成真实规律。

（4）多项式族改变了系数所对应的基；旧 poly1d 还改变了系数顺序，不能直接混用。

## 练习

（1）表示 2 − 3x + x³，并计算 x=0、1、2 处的值。解释为什么系数中必须保留一个 0。

In [14]:
points = np.array([0.0, 1.0, 2.0])
# 在此创建 Polynomial 并求值。
# 检查：手算三个位置的结果，并说明每个系数对应的次数。

（2）先预测下面结果，再运行。说明 coef 是否直接表示原变量的系数。

In [15]:
example = Polynomial([0, 1], domain=[0, 4])
print(example(np.array([0.0, 2.0, 4.0])))
print(example.convert().coef)
# 在此写出从原变量到映射变量的关系，再解释输出。

[-1.  0.  1.]
[-1.   0.5]


（3）对 3x² 求导，求从 0 到 2 的定积分。使用多项式方法完成，再用手算检查。

In [16]:
p = Polynomial([0, 0, 3])
# 在此用 deriv()、integ() 计算。
# 检查：导函数为 6x，定积分为 8。

（4）用下列三个点拟合直线。现在要求把结果交给只接受“原变量幂基系数”的程序：应直接用 fitted.coef，还是先 convert()？说明理由。若只为了让这三个点误差更小而把次数提高到 8，诊断中应该检查什么？

In [17]:
x = np.array([20.0, 21.0, 22.0])
y = np.array([41.0, 43.0, 45.0])
# 在此拟合直线，选取符合约定的系数，并重新求值核对。
# 检查：得到的原变量系数接近 [1, 2]；解释区间映射及数值秩的意义。

### 重点练习提示

对应第（4）题。先独立完成，再按需要查看提示。

（1）先看拟合对象的 domain 和 window，判断 coef 使用哪个变量。

（2）把 [20, 22] 映射到 [−1, 1]，再将映射变量的直线展开回原变量。

### 重点练习参考解析

对应第（4）题。

Polynomial.fit(x, y, 1) 默认将数据区间 [20, 22] 映射到 [−1, 1]。本题映射变量为 u=x−21，所以拟合表达式为 43+2u，fitted.coef 接近 [43, 2]。交给原变量幂基接口时，必须用 fitted.convert().coef，得到接近 [1, 2]。两种表示在原 x 上求值都应接近 [41, 43, 45]。

8 次拟合含 9 个系数，3 个观测不能提供 9 个独立约束。用 full=True 查看数值秩、奇异值和 rcond，不能仅因样本残差小就判断高次拟合可靠。应同时说明参数不足以由这三个点唯一确定。

## 参考与引用来源

本章新增示意图由 CMYK Labs 原创，依据下表对应概念与本章教学输入绘制；示意图不作为实际运行截图或数学证明。

| 网站 | 本章参考内容与定位 |
| --- | --- |
| NumPy 官方文档 | NumPy 2.5：[Polynomial](https://numpy.org/doc/2.5/reference/generated/numpy.polynomial.polynomial.Polynomial.html) 的 coef、domain、window；[Convenience Classes](https://numpy.org/doc/2.5/reference/routines.polynomials.classes.html) 的基本运算和区间映射；[fit](https://numpy.org/doc/2.5/reference/generated/numpy.polynomial.polynomial.Polynomial.fit.html) 的参数、诊断和 convert().coef；[polyfit](https://numpy.org/doc/2.5/reference/generated/numpy.polynomial.polynomial.polyfit.html) 的返回值、RankWarning 与 Notes 中的条件和高次拟合风险；[deriv](https://numpy.org/doc/2.5/reference/generated/numpy.polynomial.polynomial.Polynomial.deriv.html)、[integ](https://numpy.org/doc/2.5/reference/generated/numpy.polynomial.polynomial.Polynomial.integ.html) 的阶数、积分常数与下限；[convert](https://numpy.org/doc/2.5/reference/generated/numpy.polynomial.polynomial.Polynomial.convert.html) 的默认区间及 kind；[roots](https://numpy.org/doc/2.5/reference/generated/numpy.polynomial.polynomial.Polynomial.roots.html) 及 [polyroots](https://numpy.org/doc/2.5/reference/generated/numpy.polynomial.polynomial.polyroots.html) 的误差 Notes；[Polynomials](https://numpy.org/doc/2.5/reference/routines.polynomials.html) 的多项式族与新旧接口；[Chebyshev](https://numpy.org/doc/2.5/reference/generated/numpy.polynomial.chebyshev.Chebyshev.html) 的系数表示；[poly1d](https://numpy.org/doc/2.5/reference/generated/numpy.poly1d.html) 的降幂顺序。 |